# 02 — Analysis

Main effects in both directions, interactions against an explicit null, and
dispersion across seeds. No GPU needed.

Two rules the tool enforces, worth remembering when reading the output. Where
the from-below and from-above estimates of a main effect **disagree**, neither
may be quoted alone — the disagreement *is* the interaction. And nothing is
quotable without dispersion, so a partial campaign shows `nan` error bars and
`UNDETERMINED` rather than a confident-looking number.


## 1. Drive and repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ---------------------------------------------------------------------------
# The one place paths are defined. Everything else derives from DRIVE_ROOT.
#
#   e3dgsuw/
#     dataset/     the four scenes (original) + undistorted/  <- created below
#     dense/       M1 clouds, with SHA-256 sidecars
#     runs/        <cell>/<scene>/s<seed>/  -- one run, all of it together
#     analysis/    analyse.py output, figures, tables
#     run_ledger.json
# ---------------------------------------------------------------------------
DRIVE_ROOT   = '/content/drive/MyDrive/e3dgsuw'
DATASET_DIR  = f'{DRIVE_ROOT}/dataset'
DATA_UNDIST  = f'{DATASET_DIR}/undistorted'
DENSE_DIR    = f'{DRIVE_ROOT}/dense'
ANALYSIS_DIR = f'{DRIVE_ROOT}/analysis'

# Training reads from local disk, not Drive: the scene loader pulls every image
# at startup, and Drive's FUSE layer makes that far slower than a single copy.
LOCAL_DATA   = '/content/data'

REPO_URL  = 'https://github.com/dinanirham/An-Efficient-3D-Gaussian-Splatting-for-Underwater-3D-Reconstruction.git'
REPO_DIR  = '/content/e3dgsuw'
IMPL_DIR  = f'{REPO_DIR}/implementation'
SCENES    = ['Curasao', 'IUI3-RedSea', 'JapaneseGradens-RedSea', 'Panama']

import os
assert os.path.isdir(DRIVE_ROOT), (
    f'{DRIVE_ROOT} not found. Check the folder name, or edit DRIVE_ROOT above.')
for d in (DATA_UNDIST, DENSE_DIR, f'{DRIVE_ROOT}/runs', ANALYSIS_DIR):
    os.makedirs(d, exist_ok=True)


def find_originals():
    """Locate the four scenes under dataset/, however they were arranged.

    Accepts the scenes directly under dataset/, or nested one level (e.g.
    dataset/SeathruNeRF_dataset/). Returns the directory that contains them.
    """
    candidates = [DATASET_DIR] + [
        os.path.join(DATASET_DIR, d) for d in sorted(os.listdir(DATASET_DIR))
        if os.path.isdir(os.path.join(DATASET_DIR, d)) and d != 'undistorted'
    ]
    for base in candidates:
        if all(os.path.isdir(os.path.join(base, s)) for s in SCENES):
            return base
    return None


DATA_ORIG = find_originals()
print('drive root :', DRIVE_ROOT)
print('originals  :', DATA_ORIG or 'NOT FOUND')
print('undistorted:', DATA_UNDIST)


In [ ]:
import os, subprocess

# If the repository is private, create a fine-grained token with read access
# and set it here (or in Colab's Secrets). Leave as None for a public repo.
GITHUB_TOKEN = None

url = REPO_URL
if GITHUB_TOKEN:
    url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@')

if not os.path.exists(REPO_DIR):
    r = subprocess.run(['git','clone','--depth','1',url,REPO_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(
            'clone failed. If the repository is private, set GITHUB_TOKEN '
            f'above.\n{r.stderr[-800:]}')
else:
    subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'], check=True)

os.chdir(IMPL_DIR)
print(subprocess.run(['git','-C',REPO_DIR,'log','--oneline','-1'],
                     capture_output=True, text=True).stdout.strip())


## 2. Campaign state

In [ ]:
!python -m tools.run_ledger status --output_root "$DRIVE_ROOT"


## 3. Contrasts

Quality metrics combine additively; ratio measures (storage, primitive count,
frame rate) combine multiplicatively, in log space.


In [ ]:
!python -m tools.analyse \
    --output_root "$DRIVE_ROOT" \
    --weighting unweighted \
    --json "$ANALYSIS_DIR/analysis_unweighted.json"


In [ ]:
# Scene image counts are unequal (21/29/20/18), so the two weightings differ by
# more than many ablation differences in this literature. Reporting both
# removes an easy source of disagreement.
!python -m tools.analyse \
    --output_root "$DRIVE_ROOT" \
    --weighting image_weighted \
    --json "$ANALYSIS_DIR/analysis_image_weighted.json"


## 4. The central hypothesis (H4)

Not a between-cell comparison: the depth normalisation constants and the medium
coefficients across the simplification boundary in A2. A jump in β at 15 000 is
the signature of the identifiability failure; its absorption inside the
re-identification burst is the signature of the fix.


In [ ]:
import glob, csv
import matplotlib.pyplot as plt

paths = sorted(glob.glob(f'{DRIVE_ROOT}/runs/A2/*/s0/diagnostics.csv'))
if not paths:
    print('No A2 runs yet — that is stage S2.')
for p in paths:
    rows  = [r for r in csv.DictReader(open(p)) if r['beta_att_r']]
    if not rows:
        continue
    scene = p.split('/runs/A2/')[1].split('/')[0]
    it    = [int(r['iteration']) for r in rows]
    beta  = [float(r['beta_att_r']) for r in rows]
    zmax  = [float(r['z_max']) if r['z_max'] else float('nan') for r in rows]

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
    ax[0].plot(it, beta);  ax[0].axvline(15000, ls='--', c='r')
    ax[0].set_title(f'{scene}: beta_att (red channel)')
    ax[1].plot(it, zmax);  ax[1].axvline(15000, ls='--', c='r')
    ax[1].set_title('z_max — depth normalisation')
    for a in ax: a.set_xlabel('iteration')
    plt.tight_layout(); plt.savefig(f'{ANALYSIS_DIR}/h4_{scene}.png', dpi=140)
    plt.show()


## 5. Storage — per primitive as well as total

The codebook is a fixed cost, so the compression ratio grows with primitive
count. M2 reduces that count, so a ratio that fell because *N* fell would
otherwise read as quantization performing worse.


In [ ]:
!python -m tools.analyse --output_root "$DRIVE_ROOT" \
    --metric bytes_per_primitive --metric total_bytes --metric n_primitives_final


---
Outputs land in `analysis/` on Drive.
